# Brain Tumor MRI Image Classification

Hi! This is my capstone project for the GUVI-HCL Data Science / AI-ML course.

**Goal:** Classify brain MRI images into 4 categories (glioma, meningioma, pituitary, no tumor) using deep learning.

I am doing two things in this notebook:
1. Building my own small CNN from scratch (I want to see how well a "basic" model does).
2. Using transfer learning with a pretrained model (MobileNetV2) to try and get better accuracy.

At the end I compare both models and save the better one so I can use it in my Streamlit app.

**Dataset:** I used the "Brain Tumor MRI Dataset" (the one shared in the project doc, downloaded from Kaggle). 
It already comes split into `Training` and `Testing` folders, and inside each of those there are 4 subfolders 
(one per class): `glioma`, `meningioma`, `pituitary`, `notumor`.

Folder structure I am assuming:
```
dataset/
    Training/
        glioma/
        meningioma/
        notumor/
        pituitary/
    Testing/
        glioma/
        meningioma/
        notumor/
        pituitary/
```


## Step 0 - Importing all the libraries I will need

In [ ]:
# basic libraries for handling arrays and plotting
import numpy as np                     # for numerical operations on arrays
import pandas as pd                    # just in case I need to make small tables
import matplotlib.pyplot as plt        # to plot images and graphs
import seaborn as sns                  # to make my confusion matrix look nicer
import os                              # to work with folders and file paths
import random                          # to pick random images to display

# deep learning libraries (I am using TensorFlow / Keras)
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator   # for loading + augmenting images
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense,
                                      Dropout, BatchNormalization, GlobalAveragePooling2D)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam

# for evaluating the model at the end
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# just to make sure my results are (mostly) the same every time I run this
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully!")

## Step 1 - Understanding the dataset

Before building any model, I want to actually look at my data first.
- How many images are in each class?
- Is the dataset balanced or imbalanced?
- What do the images actually look like?

I am setting my folder paths below. (I uploaded the dataset to my Google Drive / Colab, so change 
the path below if you are running this somewhere else.)

In [ ]:
# path to my dataset - matches the folder structure I actually got after unzipping "Tumour"
# turns out the zip extracted with an extra nested folder: Tumour/Tumour/<split>/<class>
# (checked this with os.listdir("Tumour") which showed just ['Tumour'] inside)
train_dir = "Tumour/Tumour/train"
val_dir   = "Tumour/Tumour/valid"
test_dir  = "Tumour/Tumour/test"

# these are my 4 classes, taken directly from the folder names
# (note: the "no tumor" class folder is named "no_tumor" with an underscore in my dataset)
# I also noticed my dataset has a stray "_classes.csv" file sitting in each split folder
# next to the class subfolders, so I filter to keep ONLY actual folders, not that csv file
class_names = sorted([
    name for name in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, name))
])
print("Classes found:", class_names)

In [ ]:
# let's count how many images are in each class (training set)
# this helps me check if the dataset is balanced or not

for class_name in class_names:
    folder_path = os.path.join(train_dir, class_name)
    num_images = len(os.listdir(folder_path))          # counting files in that folder
    print(f"{class_name} : {num_images} images")

# doing the same thing for the validation folder
print("\n--- Validation set ---")
for class_name in class_names:
    folder_path = os.path.join(val_dir, class_name)
    num_images = len(os.listdir(folder_path))
    print(f"{class_name} : {num_images} images")

# doing the same thing for the testing folder
print("\n--- Testing set ---")
for class_name in class_names:
    folder_path = os.path.join(test_dir, class_name)
    num_images = len(os.listdir(folder_path))
    print(f"{class_name} : {num_images} images")

In [ ]:
# quick bar chart to visualize the class distribution (training data)
counts = [len(os.listdir(os.path.join(train_dir, c))) for c in class_names]

plt.figure(figsize=(7,4))
plt.bar(class_names, counts, color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"])
plt.title("Number of Training Images per Class")
plt.xlabel("Tumor Type")
plt.ylabel("Number of Images")
plt.show()

# from the graph I can see the classes are not perfectly equal, but not too imbalanced either.
# data augmentation later should help with this a bit.

In [ ]:
# let's look at a few random sample images from each class, just to get a feel for the data
plt.figure(figsize=(12, 10))

img_num = 1
for class_name in class_names:
    folder_path = os.path.join(train_dir, class_name)
    # picking 3 random images from this class folder
    sample_files = random.sample(os.listdir(folder_path), 3)

    for file_name in sample_files:
        img_path = os.path.join(folder_path, file_name)
        img = plt.imread(img_path)

        plt.subplot(4, 3, img_num)
        plt.imshow(img, cmap="gray")
        plt.title(class_name)
        plt.axis("off")
        img_num += 1

plt.tight_layout()
plt.show()

# Note: the MRI scans look grayscale, but some files might be saved as RGB.
# I will make sure to handle this consistently when I load the data below.

## Step 2 - Data Preprocessing and Augmentation

Now I need to get the images ready for the model:
- Resize every image to the same size (I chose 224x224 because that's what most pretrained models expect).
- Normalize pixel values so they are between 0 and 1 (instead of 0-255). This helps the model train faster.
- Apply some data augmentation ONLY on the training images (rotation, zoom, flipping, etc). This is basically 
  creating "fake new" training images so my model does not just memorize the training set (overfitting).

I am using Keras' `ImageDataGenerator` because it is simple and does the resizing + augmentation for me 
automatically while training.

In [ ]:
IMG_SIZE = (224, 224)     # width and height I am resizing every image to
BATCH_SIZE = 32           # how many images the model looks at in one go

# for the TRAINING data - I add augmentation here so the model generalizes better
# (I don't need validation_split anymore since my dataset already came with a separate "valid" folder)
train_datagen = ImageDataGenerator(
    rescale=1./255,            # scales pixel values from 0-255 down to 0-1
    rotation_range=15,         # randomly rotate images by up to 15 degrees
    width_shift_range=0.1,     # randomly shift image left/right
    height_shift_range=0.1,    # randomly shift image up/down
    zoom_range=0.1,            # randomly zoom in a little
    horizontal_flip=True,      # randomly flip images horizontally
    brightness_range=[0.9, 1.1]   # slightly change brightness
)

# for VALIDATION and TEST data - no augmentation here! we only rescale, since we want
# to check the model on real, unmodified images
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
# this generator will feed batches of augmented training images to the model
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",   # because we have more than 2 classes
    shuffle=True,
    seed=42
)

# this generator gives us the validation images (from the separate "valid" folder in my dataset)
val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=42
)

# this generator is for the final, untouched test set
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False   # keep shuffle False here so I can match predictions back to true labels later
)

# just double-checking the class index mapping Keras created for me
print("Class indices:", train_generator.class_indices)

## Step 3 - Building my own CNN from scratch

Before jumping to pretrained models, I wanted to try building a simple Convolutional Neural Network 
myself, just using Conv2D and MaxPooling2D layers stacked together, followed by some Dense layers 
at the end for the actual classification.

I added `BatchNormalization` and `Dropout` because in class we learned these help the model train more 
stably and avoid overfitting.

In [ ]:
num_classes = len(class_names)   # should be 4 in our case

# building the model layer by layer using Sequential API
custom_cnn = Sequential([

    # ---- Block 1 ----
    Conv2D(32, (3,3), activation="relu", input_shape=(224, 224, 3)),  # 32 filters, 3x3 kernel
    BatchNormalization(),          # normalizes outputs, helps training go smoother
    MaxPooling2D(pool_size=(2,2)), # reduces the image size by half

    # ---- Block 2 ----
    Conv2D(64, (3,3), activation="relu"),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),

    # ---- Block 3 ----
    Conv2D(128, (3,3), activation="relu"),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),

    # ---- Block 4 ----
    Conv2D(128, (3,3), activation="relu"),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),

    # now flatten the 3D feature maps into a 1D vector so Dense layers can use them
    Flatten(),

    Dense(256, activation="relu"),
    Dropout(0.4),        # randomly turns off 40% of neurons during training, helps avoid overfitting

    Dense(num_classes, activation="softmax")   # final output layer - one node per class
])

# printing the model summary so I can see the layers and number of parameters
custom_cnn.summary()

In [ ]:
# compiling the model - telling Keras how to train it
custom_cnn.compile(
    optimizer=Adam(learning_rate=0.0001),   # Adam optimizer, small learning rate so training is stable
    loss="categorical_crossentropy",        # standard loss for multi-class classification
    metrics=["accuracy"]
)

In [ ]:
# setting up callbacks so training stops automatically if it's not improving,
# and so I always save the best version of the model (not just the last epoch)

early_stop_cnn = EarlyStopping(
    monitor="val_loss",
    patience=5,                # wait 5 epochs before stopping if val_loss doesn't improve
    restore_best_weights=True
)

checkpoint_cnn = ModelCheckpoint(
    "custom_cnn_best.h5",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

In [ ]:
# training the custom CNN
# NOTE: this can take a while depending on how many images you have / if you have a GPU

history_cnn = custom_cnn.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    callbacks=[early_stop_cnn, checkpoint_cnn]
)

In [ ]:
# plotting accuracy and loss curves so I can see how training went

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history_cnn.history["accuracy"], label="Train Accuracy")
plt.plot(history_cnn.history["val_accuracy"], label="Validation Accuracy")
plt.title("Custom CNN - Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history_cnn.history["loss"], label="Train Loss")
plt.plot(history_cnn.history["val_loss"], label="Validation Loss")
plt.title("Custom CNN - Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## Step 4 - Transfer Learning with MobileNetV2

Now let's try transfer learning. Instead of learning everything from scratch, I am using MobileNetV2, 
which was already trained on the huge ImageNet dataset. The idea is that it already knows how to 
detect basic shapes/edges/textures, so I just need to teach it the specific brain tumor classes.

Steps I followed:
1. Load MobileNetV2 without its original top (classification) layer.
2. Freeze all its layers so we don't mess up the pretrained weights.
3. Add my own small classification head on top for our 4 classes.
4. Train only the new head first.

In [ ]:
# loading MobileNetV2 with weights trained on ImageNet
# include_top=False means we don't include their original 1000-class classification layer
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# freezing the base model so its weights don't get updated during initial training
base_model.trainable = False

print("Number of layers in MobileNetV2 base:", len(base_model.layers))

In [ ]:
# building my new model on top of the frozen base
inputs = base_model.output
x = GlobalAveragePooling2D()(inputs)     # converts each feature map into a single number (better than Flatten here)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(num_classes, activation="softmax")(x)

transfer_model = Model(inputs=base_model.input, outputs=outputs)

transfer_model.summary()

In [ ]:
transfer_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop_tl = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
checkpoint_tl = ModelCheckpoint("transfer_model_best.h5", monitor="val_accuracy",
                                 save_best_only=True, verbose=1)

In [ ]:
# training just the new head layers first (base model is still frozen)
history_tl = transfer_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=[early_stop_tl, checkpoint_tl]
)

### Fine-tuning

Now that the new top layers have learned something useful, I am going to "unfreeze" the last few layers 
of MobileNetV2 and train them a little bit too, with a very small learning rate. This is called 
fine-tuning and usually gives a small extra boost in accuracy.

In [ ]:
# unfreezing the base model
base_model.trainable = True

# but I only want to fine-tune the LAST 30 layers, keep the earlier ones frozen
# (earlier layers learn very generic features like edges, we don't want to disturb those)
fine_tune_at = len(base_model.layers) - 30

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# recompiling with a much smaller learning rate since we are fine-tuning
transfer_model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# training for a few more epochs
history_finetune = transfer_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stop_tl, checkpoint_tl]
)

In [ ]:
# plotting accuracy/loss for the transfer learning model (initial training + fine-tuning combined)
acc = history_tl.history["accuracy"] + history_finetune.history["accuracy"]
val_acc = history_tl.history["val_accuracy"] + history_finetune.history["val_accuracy"]
loss = history_tl.history["loss"] + history_finetune.history["loss"]
val_loss = history_tl.history["val_loss"] + history_finetune.history["val_loss"]

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(acc, label="Train Accuracy")
plt.plot(val_acc, label="Validation Accuracy")
plt.title("Transfer Learning (MobileNetV2) - Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1,2,2)
plt.plot(loss, label="Train Loss")
plt.plot(val_loss, label="Validation Loss")
plt.title("Transfer Learning (MobileNetV2) - Loss")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

## Step 5 - Model Evaluation

Time to check how well both models actually do on the TEST set (images the models have never seen).
I will look at:
- Overall accuracy
- Precision, Recall, F1-score for each class (classification report)
- Confusion matrix (to see which classes get confused with each other)

In [ ]:
def evaluate_model(model, generator, model_name):
    """Small helper function I wrote so I don't have to repeat this code twice
    (once for the custom CNN, once for the transfer learning model)."""

    # getting predictions (probabilities) for the whole test set
    predictions = model.predict(generator)
    predicted_classes = np.argmax(predictions, axis=1)   # picking the class with highest probability

    true_classes = generator.classes                     # actual labels
    labels = list(generator.class_indices.keys())         # class names in the right order

    print(f"===== {model_name} =====")
    print("Test Accuracy:", accuracy_score(true_classes, predicted_classes))
    print("\nClassification Report:\n")
    print(classification_report(true_classes, predicted_classes, target_names=labels))

    # confusion matrix
    cm = confusion_matrix(true_classes, predicted_classes)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    return accuracy_score(true_classes, predicted_classes)

In [ ]:
# evaluating the custom CNN
cnn_accuracy = evaluate_model(custom_cnn, test_generator, "Custom CNN")

In [ ]:
# evaluating the transfer learning model
# need to reset the generator first so predictions line up with the true labels correctly
test_generator.reset()
transfer_accuracy = evaluate_model(transfer_model, test_generator, "Transfer Learning (MobileNetV2)")

## Step 6 - Comparing both models

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["Custom CNN", "Transfer Learning (MobileNetV2)"],
    "Test Accuracy": [cnn_accuracy, transfer_accuracy]
})

print(comparison_df)

plt.figure(figsize=(6,4))
plt.bar(comparison_df["Model"], comparison_df["Test Accuracy"], color=["#4C72B0", "#55A868"])
plt.ylim(0, 1)
plt.title("Model Comparison - Test Accuracy")
plt.ylabel("Accuracy")
for i, v in enumerate(comparison_df["Test Accuracy"]):
    plt.text(i, v + 0.02, f"{v:.2f}", ha="center")
plt.show()

# from my results, the transfer learning model usually does better than the custom CNN,
# which makes sense since it already had a head start from being trained on ImageNet.

In [ ]:
# saving both trained models so I can load them later in my Streamlit app
custom_cnn.save("custom_cnn_model.h5")
transfer_model.save("transfer_learning_model.h5")

print("Models saved! I will use whichever one had the higher accuracy in my Streamlit app.")

## Conclusion

- I built a custom CNN from scratch and also used transfer learning with MobileNetV2.
- Transfer learning gave better accuracy in my case, which is what I expected since MobileNetV2 already 
  "knows" a lot about images in general from ImageNet.
- I saved both models as `.h5` files so I can load the best one inside my Streamlit app (`app.py`) and 
  let users upload their own MRI image to get a prediction.

**Next steps / things I could improve if I had more time:**
- Try other pretrained models like ResNet50 or EfficientNetB0 and compare.
- Try to fix the class imbalance a bit better using class weights.
- Collect more MRI images so the model becomes more robust.
